# Initialize Git Repository
https://docs.ycrc.yale.edu/clusters-at-yale/guides/github/

In [ ]:
#git init -b main  # Initializes the repo with 'main' as the default branch
#git add .
#git commit -m "Initial commit"
#git remote add origin git@github.com:innacohen/mod-extract.git
#git push -u origin main --force

# Download mod files as a single JSON file

In [ ]:
import os
import pandas as pd
import requests
import json
import re
from tqdm import tqdm

# Load dataset from Excel
df = pd.read_excel("../data/model_db_annotations.xlsx")

# Output JSON files
output_json = "../data/mod_files.json"
failed_json = "../data/failed_downloads.json"

# Lists to store mod file data and failed downloads
mod_files_data = []
failed_downloads = []

# Function to convert ModelDB URL to direct download URL
def get_direct_download_url(url):
    match = re.search(r"https://modeldb\.science/(\d+)\?tab=2&file=(.+)", url)
    if match:
        model_id, file_path = match.groups()
        return f"https://modeldb.science/getModelFile?model={model_id}&file={file_path}", file_path
    return None, None  # Return None if the URL doesn't match the expected pattern

# Function to fetch .mod file content and store in JSON
def fetch_mod_file(row_id, file_hash, raw_sha, count, url):
    direct_url, file_path = get_direct_download_url(url)
    
    # Default structure for JSON entry
    file_entry = {
        "row_id": row_id,
        "file_hash": file_hash,
        "raw_sha": raw_sha,
        "count": count,
        "url": url,
        "download_url": direct_url,
        "content": None,  # Default to None (indicating missing content)
        "error_code": None  # New field for failed downloads
    }

    if not direct_url:
        print(f"🚨 Skipping invalid URL: {url}")
        file_entry["error_code"] = "Invalid URL"
        failed_downloads.append(file_entry)
        mod_files_data.append(file_entry)  # Store even failed ones
        return

    try:
        response = requests.get(direct_url, timeout=10)
        response.raise_for_status()
        file_entry["content"] = response.text  # Store the downloaded content
    except requests.exceptions.HTTPError as http_err:
        file_entry["error_code"] = response.status_code  # Store specific HTTP error code
        print(f"❌ Failed to fetch {direct_url} - HTTP {response.status_code}: {http_err}")
        failed_downloads.append(file_entry)
    except requests.exceptions.RequestException as e:
        file_entry["error_code"] = "Request Error"
        print(f"❌ Failed to fetch {direct_url}: {e}")
        failed_downloads.append(file_entry)

    mod_files_data.append(file_entry)  # Store the file entry (success or failure)

# Select rows from 467 (corresponding to row_id 468) and take 11 rows
start = 0
n_rows = df.shape[0] 
df_subset = df.iloc[start:start+n_rows]

# Process selected files
for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Fetching .mod files"):
    fetch_mod_file(
        row["row_id"], 
        row["file_hash"], 
        row["raw_sha"], 
        row["count"],  
        row["url"]
    )

# Save all .mod data to JSON (including failed downloads)
with open(output_json, "w", encoding="utf-8") as json_file:
    json.dump(mod_files_data, json_file, indent=4)

# Save failed downloads separately for easy retrying
if failed_downloads:
    with open(failed_json, "w", encoding="utf-8") as json_file:
        json.dump(failed_downloads, json_file, indent=4)
    print(f"⚠️ Some downloads failed. Check {failed_json} for details.")

print(f"\n✅ All .mod files (including failures) saved in {output_json}")

# Read the Training Data

In [1]:
import pandas as pd
import numpy as np

In [20]:
#Converted to long format in R
df_raw = pd.read_csv("../data/annotations_long.csv")
keep = ["row_id", "file_hash","type","subtype_confidence","notes_free_text","label"]
df_anno = df_raw[keep].copy()

In [21]:
df_anno

,row_id,file_hash,type,subtype_confidence,notes_free_text,label
0,1,f8be35d0c20d1b1f3de4c44323e1780ee24f06893b6364...,I Na,3 - Highly confident,NaN,i_na_persistent
1,2,e97ca8a7f9734805832e5ae75442d19d3d7796b9a24190...,I K,2 - Mildly confident,STATES n l taul = 0.26*(v+50)/qtl,i_k_a_type_slow
2,4,606423f8f4a4f406f3387c7ee6f142bd121237272c347d...,Receptor,3 - Highly confident,NaN,r_gaba_type_a
3,5,99713c0032634e96cc7cde2dce02d8aa2baf75ad4cc835...,Receptor,3 - Highly confident,NaN,r_gaba_general
4,6,2d34ae241d628e7f25374c1bd84a8138cedc6a54689568...,Receptor,3 - Highly confident,mix,r_glutamate_ampa
...,...,...,...,...,...,...
414,462,ed87e94223dec25e43020f389207c0e362361efd05f3b0...,I Na,3 - Highly confident,NaN,i_na_transient
415,463,414840eea6fe6455d3377f99f43543155d2e4771b8cf8c...,I Ca,3 - Highly confident,NaN,i_ca_t_type_lt
416,464,43cdc726cbc3bc20eb08263de0028f8bfba35edba720eb...,Exclude,3 - Highly confident,NaN,exclude_accumulation_mechanism
417,467,abd4902abc5ad78269b6b9a4cc6d430bdd2bdd777cafd2...,I K,3 - Highly confident,NaN,i_k_delayed_rectifier


# Feature Engineering

In [4]:
import os
import pandas as pd
import requests
import json
import re
from bs4 import BeautifulSoup
from tqdm import tqdm
from sklearn.preprocessing import MultiLabelBinarizer
pd.set_option("display.max_columns", None)

In [5]:
def View(df, rows=None, cols=None, width=None):
    """Displays the first `rows` of the DataFrame like R's View() by adjusting Pandas settings."""
    
    # Show only the first `rows` of the DataFrame
    with pd.option_context(
        "display.max_rows", rows,  # Limit number of rows shown
        "display.max_columns", cols,  # Show all columns
        "display.max_colwidth", width,  # Show full column width
        "display.expand_frame_repr", False  # Prevent column wrapping
    ):
        display(df.head(rows))  # Show only the first `rows`


In [6]:
# Function to extract mod directory from the URL
def get_dir(url):
    match = re.search(r"file=([^/]+)/[^/]+\.mod", url)  # Extract the directory name before the .mod file
    return match.group(1) if match else None  # Return directory name if found, else None

# Function to extract mod file name without extension
def get_fname(url):
    match = re.search(r"/([^/]+)\.mod$", url)  # Get filename without extension
    return match.group(1) if match else None  # Return only the name (e.g., 'na')

# Function to extract model_id from the URL
def get_model_id(url):
    match = re.search(r"https://modeldb\.science/(\d+)", url)
    return int(match.group(1)) if match else None  # Convert to integer

# Function to extract all TITLE occurrences from .mod content
def get_title(content):
    if pd.isna(content):  
        return None
    matches = re.findall(r"^TITLE\s+([^\n:]+)", content, re.MULTILINE)  # Stop at comments
    return matches if matches else None

# Function to extract all COMMENT sections from .mod content
def get_comment(content):
    if pd.isna(content):  
        return None
    matches = re.findall(r"COMMENT\s+(.*?)(?:\s+ENDCOMMENT|\Z)", content, re.DOTALL)  
    return matches if matches else None

# Function to extract all SUFFIX occurrences from .mod content
def get_suffix(content):
    if pd.isna(content):  
        return None
    matches = re.findall(r"SUFFIX\s+([^\n:\s]+)", content, re.MULTILINE)  # Stop at comments
    return matches if matches else None


def get_use_ion(content):
    """
    Extracts the ion names used in the 'USEION' statements from NEURON mod file content.

    Parameters:
    - content (str): The content of the .mod file.

    Returns:
    - list: A list of ions used in 'USEION' statements, or None if none are found.
    """
    if pd.isna(content):  
        return None
    
    # Find all occurrences of USEION followed by an ion name
    matches = re.findall(r"USEION\s+(\w+)", content, re.MULTILINE)

    return matches if matches else None


# Function to extract all ions listed after READ but stopping before WRITE, USEION, RANGE, GLOBAL, NONSPECIFIC_CURRENT, or VALENCE
def get_read_ion(content):
    if pd.isna(content):  
        return None
    
    matches = re.findall(r"USEION\s+\w+\s+READ\s+([\w,\s]+?)(?=\s+(?:WRITE|USEION|RANGE|GLOBAL|NONSPECIFIC_CURRENT|VALENCE|:|\n|$))", content, re.MULTILINE)

    if not matches:
        return None

    read_ions = [ion.strip() for match in matches for ion in re.split(r"[,\s]+", match) if ion]

    return read_ions if read_ions else None  


# Function to extract all ions listed after WRITE, stopping before VALENCE
def get_write_ion(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"WRITE\s+([^\n:]+?)(?=\s+(?:VALENCE|:|\n|$))", content, re.MULTILINE)

    if not matches:
        return None

    write_ions = [ion.strip() for match in matches for ion in re.split(r"[,\s]+", match) if ion]

    return write_ions if write_ions else None  


def write_current_yn(ions):
    """
    Checks if mod_write_ion contains an ion that starts with 'i' (indicating a current).

    Args:
        write_ions (list): List of ions written in the mod file.

    Returns:
        int: 1 if any ion starts with 'i', otherwise 0.
    """
    if not ions:  # Handle empty lists or None
        return 0

    return int(any(ion.startswith("i") for ion in ions))


# Function to extract all NONSPECIFIC currents
def get_nonspecific_current(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"NONSPECIFIC_CURRENT\s+([^\n:]*)", content)

    if not matches:
        return None

    nonspecific_currents = [curr.strip() for match in matches for curr in re.split(r"[,\s]+", match) if curr]

    return nonspecific_currents if nonspecific_currents else None  

#todo: should we assume we only want active variables or also extract ones that are commented out?
# Function to extract RANGE variables based on mode
def get_range(content, mode="active"):
    if pd.isna(content):
        return None  # Return None if content is missing

    # Extract active RANGE variables (not commented out)
    active_matches = re.findall(r"^\s*RANGE\s+([\w\s,]+)", content, re.MULTILINE)

    # Extract commented-out RANGE variables (lines starting with ": RANGE")
    commented_matches = re.findall(r"^\s*:\s*RANGE\s+([\w\s,]+)", content, re.MULTILINE)

    # Process active RANGE variables
    active_vars = [var.strip() for match in active_matches for var in re.split(r"[,\s]+", match) if var]

    # Process commented-out RANGE variables
    commented_vars = [var.strip() for match in commented_matches for var in re.split(r"[,\s]+", match) if var]

    if mode == "active":
        return active_vars if active_vars else None
    elif mode == "commented":
        return commented_vars if commented_vars else None
    elif mode == "all":
        return {"active": active_vars if active_vars else None, "commented": commented_vars if commented_vars else None}
    else:
        raise ValueError("Invalid mode! Choose from 'all', 'active', or 'commented'.")


# Function to extract only active RANGE variables, stopping at colons and the end of the line
def get_range(content):
    if pd.isna(content):
        return None  # Return None if content is missing

    # Extract all RANGE statements (each line separately), stopping before colons
    matches = re.findall(r"^\s*RANGE\s+([^\n:]*)", content, re.MULTILINE)

    if not matches:
        return None

    # Process active RANGE variables, ensuring they don't capture anything past the colon
    active_vars = [var.strip() for match in matches for var in re.split(r"[,\s]+", match) if var]

    return active_vars if active_vars else None  # Return only active variables
    
# Function to extract parameter names and values as a dictionary
def get_parameter(content):
    if pd.isna(content):  
        return None
    
    matches = re.findall(r"PARAMETER\s*\{([^}]*)\}", content, re.MULTILINE)

    if not matches:
        return None

    param_dict = {}
    
    for match in matches:
        for line in match.split("\n"):
            line = line.strip()
            if line.startswith(":"):  # Ignore commented-out lines
                continue
            param_match = re.match(r"(\w+)\s*=\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)", line)
            if param_match:
                param_name, param_value = param_match.groups()
                param_dict[param_name] = float(param_value)  

    return param_dict if param_dict else None  

# Function to extract only active STATE variables, ignoring comments (`:`) and unit values `(mV)`, etc.
def get_state(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"STATE\s*\{([^}]*)\}", content, re.MULTILINE)

    if not matches:
        return None

    state_vars = []
    for match in matches:
        for line in match.split("\n"):
            line = line.strip()
            if line.startswith(":"):  # Ignore fully commented-out lines
                continue
            line = re.split(r"\s*:\s*", line)[0]  # Remove inline comments (anything after `:`)
            clean_line = re.sub(r"\([^)]*\)", "", line).strip()  # Remove unit values
            if clean_line:
                state_vars.append(clean_line)

    return state_vars if state_vars else None  


# Function to extract only active GLOBAL variables, ignoring commented-out (`:`) ones
def get_global(content):
    if pd.isna(content):  
        return None

    matches = re.findall(r"^\s*GLOBAL\s+([^\n:]*)", content, re.MULTILINE)

    if not matches:
        return None

    global_vars = [var.strip() for match in matches for var in re.split(r"[,\s]+", match) if var]

    return global_vars if global_vars else None  


def get_net_receive(content):
    """
    Extracts all NET_RECEIVE block arguments from MOD file content.

    Args:
        content (str): The text content of the MOD file.

    Returns:
        list or None: A list of extracted NET_RECEIVE arguments, or None if not found.
    """
    if pd.isna(content):  # Handle missing content
        return None

    # Find all occurrences of NET_RECEIVE and extract arguments
    matches = re.findall(r"^\s*NET_RECEIVE\s*\(\s*([\w, ]+)\s*\)", content, re.MULTILINE)

    if not matches:
        return None

    net_receive_vars = [var.strip() for match in matches for var in re.split(r"[,\s]+", match) if var]

    return net_receive_vars if net_receive_vars else None

#todo: modify pipeline so that if get_include points to a file that file will be included in the content too
def get_include(content):
    """
    Extracts the filename in the INCLUDE statement from MOD file content, ignoring comments.

    Args:
        content (str): The text content of the MOD file.

    Returns:
        list or None: A list of extracted INCLUDE filenames, or None if not found.
    """
    if pd.isna(content):  # Handle missing content
        return None

    # Find all occurrences of INCLUDE, ensuring commented-out ones (with ':') are ignored
    matches = re.findall(r"^\s*INCLUDE\s+\"([^\"]+)\"", content, re.MULTILINE)

    return matches if matches else None


def get_point_process(content):
    """
    Extracts the POINT_PROCESS name from MOD file content.

    Args:
        content (str): The text content of the MOD file.

    Returns:
        str or None: The extracted POINT_PROCESS name, or None if not found.
    """
    if pd.isna(content):  # Handle missing content
        return None

    # Extract the POINT_PROCESS name, ignoring comments
    match = re.search(r"^\s*POINT_PROCESS\s+([^\n:]+)", content, re.MULTILINE)

    return match.group(1).strip() if match else None


    
# Function to extract webpage heading
def get_heading(url):
    try:
        response = requests.get(url, timeout=10)  # Fetch the webpage
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Try extracting heading from the most relevant tag
        heading = soup.find("h1")  # Look for <h1> (main title)
        return heading.text.strip() if heading else None  # Return text or None
    except requests.exceptions.RequestException:
        return None  # Return None if the request fail

# Function to extract citation (text inside parentheses)
def get_citation(heading):
    if pd.isna(heading):
        return None
    match = re.search(r"\(([^)]+)\)", heading)  # Find text inside parentheses
    return match.group(1) if match else None  # Extract citation


# Function to extract first author(s) (removes "et al." and "al" correctly)
def get_author(citation):
    if pd.isna(citation):
        return None

    # Extract first author(s) before "et al" or variants
    match = re.search(r"^([\w\s&\-,]+?)(?:\s+et\s+al\.?|et)?(?:,|\s|$)", citation)  
    first_author = match.group(1).strip() if match else None  

    # Remove any trailing "al" left behind
    if first_author:
        first_author = re.sub(r"\b(al)\b", "", first_author, flags=re.IGNORECASE).strip()

    return first_author

# Function to extract the first year from citation (including shortened years like '20)
def get_year(citation):
    if pd.isna(citation):
        return None
    match = re.search(r"\b(19|20)?\d{2}\b|'\d{2}", citation)  # Find 4-digit or short year ('20)
    if match:
        return match.group(0).replace("'", "")  # Remove apostrophe but keep year as '20' if short
    return None  # Return None if no year found




def has_x86(url):
    """
    Checks if the given URL contains 'x86_64'.

    Parameters:
    - url (str): The URL to check.

    Returns:
    - int: 1 if 'x86_64' is present in the URL, 0 otherwise.
    """
    if pd.isna(url):  
        return None  # Return None for missing URLs
    return 1 if "x86_64" in url else 0


def has_error(error_code):
    """
    Checks if the given error code is not NaN (i.e., an error is present).

    Parameters:
    - error_code: The error code to check.

    Returns:
    - int: 1 if an error code is present (not NaN), 0 otherwise.
    """
    return 1 if error_code is not None else 0

def has_electrode_or_clamp(mod_name, content):
    """
    Checks whether 'clamp' is present in the mod file name OR 
    'ELECTRODE_CURRENT' is present in the NEURON mod file content.

    Parameters:
    - mod_name (str): The name of the .mod file.
    - content (str): The content of the .mod file.

    Returns:
    - int: 1 if either 'clamp' is in the mod name OR 'ELECTRODE_CURRENT' is in the content, 0 otherwise.
    """
    if pd.isna(mod_name) and pd.isna(content):
        return None  # Return None if both are missing

    has_clamp = bool(re.search(r"clamp", str(mod_name), re.IGNORECASE)) if pd.notna(mod_name) else False
    has_electrode = bool(re.search(r"\bELECTRODE_CURRENT\b", str(content))) if pd.notna(content) else False

    return 1 if has_clamp or has_electrode else 0

def map_ion(value):
    value = value.lower()  # Normalize to lowercase

    # Define regex-based categorization rules
    patterns = [
        (r'ca.*i$', 'ca_i'),
        (r'ca.*o$', 'ca_o'),
        (r'cl.*i$', 'cl_i'),
        (r'cl.*o$', 'cl_o'),
        (r'k.*i$', 'k_i'),
        (r'k.*o$', 'k_o'),
        (r'na.*i$', 'na_i'),
        (r'na.*o$', 'na_o'),
        (r'hco3.*i$', 'other_i'),
        (r'hco3.*o$', 'other_o'),
        (r'mgi$', 'mg_i'),  
        (r'mgo$', 'mg_o'),  
        (r'^img$', 'i_mg'),  
        (r'^emg$', 'e_mg'),
        (r'^e.*ca', 'e_ca'),
        (r'^e.*k', 'e_k'),
        (r'^e.*na', 'e_na'),
        (r'^e.*mg', 'e_mg'),
        (r'^e.*', 'e_other'),
        (r'^i.*ca', 'i_cal'),
        (r'^i.*k', 'i_k'),
        (r'^i.*cl', 'i_cl'),
        (r'^i.*na$', 'i_na'),  # FIX: Ensure "ina" is classified as "i_na"
        (r'^i.*mg', 'i_mg'),
        (r'^i.*', 'i_other'),
        (r'.*i$', 'other_i'),  # General rule: Anything ending in "i" is "other_i"
        (r'.*o$', 'other_o')   # General rule: Anything ending in "o" is "other_o"
    ]
    # Apply the regex patterns
    for pattern, category in patterns:
        if re.search(pattern, value):
            return category

    return "unknown"  # Default category if no match is found

def count_states(df, column_name="state"):
    """Counts the number of states in each row of the specified column."""
    df["count_states"] = df[column_name].apply(lambda x: len(x) if isinstance(x, list) else 0)
    return df

def map_new_label(value):
    if value in [
        'exclude_utility', 'exclude_non_neural', 'exclude_clamp',
        'exclude_accumulation_mechanism', 'exclude_artificial_cell',
        'exclude_gap_junction', 'exclude_localized_reactions'
    ]:
        return 'neither' 
    
    elif value in [
        'i_k_delayed_rectifier', 'r_glutamate_ampa', 'i_na_transient',
        'i_ca_l_type_ht', 'r_glutamate_nmda', 'i_other_mixed',
        'i_ca_t_type_lt', 'i_h_hyperpolarization_activated',
        'i_na_persistent','i_k_m_type'
    ]:
        return value
    elif value in [
        'i_ca_ca_pump', 'i_ca_calcium_activiated_non_specific',
        'i_ca_general', 'i_ca_p_q_type', 'i_ca_q_type', 'i_ca_r_type'
    ]:
        return 'i_ca'

    elif value in [ 'i_k_a_type_transient',    'i_k_a_type_slow']:
        return 'i_k_a'
    elif value in [
        'i_k_ca_activated_general_bk_ik_sk_and_i_ahp_currents', 'i_k_ca_activated_bk',
        'i_k_ahp', 'i_k_ca_activated_sk'
    ]:
        return 'i_k_ca'
    
    elif value in [
        'i_k_c_type', 'i_k_general', 'i_k_ht', 'i_k_ir',
        'i_k_lt', 'i_k_resistent_persistent', 'i_kx_photoreceptor'
    ]:
        return 'i_k'
    elif value in ['i_na_slow_inactivation', 'i_na_general'
    ]:
        return 'i_na'
    elif value in ['i_na_leak','i_leak_general','i_cl_leak', 'i_k_leak']:
        return 'i_leak'
    elif value in [
        'i_other_channelrhodopsin', 'i_other_cng', 'i_other_kcc2_transporter',
        'i_other_modulator_activated', 'i_other_na_ca_exchanger',
        'i_other_na_k_pump','i_nonspecific'
    ]:
        return 'i_other_mixed'
    
    elif value in ['r_gaba_type_a', 'r_gaba_general', 'r_gaba_type_b']:
        return 'r_gaba'
    
    elif value in ['r_acetylcholine_general', 'r_amino_acid_glutamate']:
        return 'r_other'
    
    return 'neither'

def map_broad_label(value):
    if value == "I K":
        return "i_k"
    elif value == "Exclude":
        return "neither"
    elif value == "Receptor":
        return "receptor"
    elif value == "I Na":
        return "i_na"
    elif value == "I Ca":
        return "i_ca"
    elif value in ["I Other/Mixed","I Cl","I H"]:
        return "i_other"
    else:
        return "check"  # Default label for unknown values

def get_tau(param_dict):
    if not isinstance(param_dict, dict):
        return None, None  # Return separate None values for direct unpacking

    # Extract values where the key contains 'tau'
    tau_values = [v for k, v in param_dict.items() if 'tau' in k.lower()]
    
    # If no tau values found, return (None, None)
    if not tau_values:
        return None, None
    
    # Compute min and max
    return min(tau_values), max(tau_values)


def get_e(param_dict):
    if not isinstance(param_dict, dict):
        return [None, None]  # Handle cases where the value is not a dictionary

    #todo: modify the v pattern so it takes like 2 characters max
    # Regex pattern to match variations of reversal potential names
    pattern = re.compile(r"^(e|rev|v|shift).*", re.IGNORECASE)

    # Extract values where the key matches the pattern
    e_values = [v for k, v in param_dict.items() if pattern.match(k)]

    # If no values found, return [None, None]
    if not e_values:
        return [None, None]

    # Compute min and max reversal potential
    return min(e_values), max(e_values)



def has_mg(content):
    """
    Checks if 'mg' appears anywhere in the given content.

    Args:
        content (str): The text content to search.

    Returns:
        int: 1 if 'mg' is found, 0 otherwise.
    """
    if pd.isna(content):  # Handle missing content
        return 0

    return int(bool(re.search(r"mg", content, re.IGNORECASE)))  # Convert Boolean to int


In [23]:
# Load JSON into DataFrame
df_all = pd.read_json("../data/mod_files.json")
#df_all.set_index("row_id", inplace=True)
# Add exclusion criteria as flags
df_all["exclude_error"] = df_all["error_code"].apply(has_error)
df_all["exclude_x86"] = df_all["url"].apply(has_x86)
df_all['dataset'] = df_all['file_hash'].isin(df_train['file_hash']).map({True: 'train', False: 'test'})
df_all2 = df_all.merge(df_anno, on=["row_id","file_hash"], how="left")
df_all2["new_label"] = df_all2["label"].map(map_new_label)

In [ ]:
#
#df_train2.loc[df_train2["exclude_x86"]==1].index == df_train2.loc[df_train2["label"]=="exclude_old_architecture"].index
#df_train2["file_hash"].nunique()
#df_train3 = df_train2.loc[(df_train2["exclude_error"] != 1) & (df_train2["exclude_x86"] != 1)] 
#df_train3["file_hash"].nunique()
#All the "exclude" labels need to be collapsed to "neither"
#df_train4["label"].value_counts()
#df_train3[df_train3["type"]=="Exclude"].index
#subtype_counts = pd.DataFrame(df_train4["label"].value_counts()).reset_index()
#list(subtype_counts[subtype_counts['count']>=10]['label'])
#subtype_counts[subtype_counts["count"] >= 10].sort_values(by="label").reset_index(drop=True)
#subtype_counts["new_label"] = subtype_counts["label"].map(map_new_label)
#subtype_counts["new_label"].unique()
#subtype_counts.groupby(["new_label","label"]).count()
#df['broad_label'] = df["type"].map(map_broad_label)


In [24]:
df = df_all2.copy()

In [25]:
# Extract features from url
#df["heading"] = df["url"].apply(get_heading)  # Extract heading from webpage
#df["citation"] = df["heading"].apply(get_citation)
#df["first_author"] = df["citation"].apply(get_author)
#df["year"] = df["citation"].apply(get_year)  # Now handles multiple years

#df["dir"] = df["url"].apply(get_dir)
df["mod_name"] = df["url"].apply(get_fname)
#df["model_id"] = df["url"].apply(get_model_id)

#  Extract features from content
#df["title"] = df["content"].apply(get_title)
#df["comment"] = df["content"].apply(get_comment)
df["suffix"] = df["content"].apply(get_suffix)
#df["use_ion"] = df["content"].apply(get_use_ion)
df["read_ion"] = df["content"].apply(get_read_ion)
#Use empty list since cannot handle None
df['read_ion2'] = df['read_ion'].apply(lambda ion_list: list(set(map_ion(ion) for ion in ion_list)) if isinstance(ion_list, list) else [])
df["write_ion"] = df["content"].apply(get_write_ion)
df['write_ion2'] = df['write_ion'].apply(lambda ion_list: list(set(map_ion(ion) for ion in ion_list)) if isinstance(ion_list, list) else [])

#df["range"] = df["content"].apply(get_range)
#df["global"] = df["content"].apply(get_global)
df["parameter"] = df["content"].apply(get_parameter)
df["state"] = df["content"].apply(get_state)
df["net_receive"] = df["content"].apply(get_net_receive)
df["point_process"] = df["content"].apply(get_point_process)
df["nonspecific_current"] = df["content"].apply(get_nonspecific_current)

In [26]:
#Numeric Features
#df["tau_min"], df["tau_max"] = zip(*df["parameter"].apply(get_tau))
#df["e_min"], df["e_max"] = zip(*df["parameter"].apply(get_e))

#Discrete features
df["states_count"] = df["state"].apply(lambda x: len(x) if isinstance(x, list) else 0)

#Binary Features
df["clamp_yn"] = df.apply(lambda row: has_electrode_or_clamp(row["mod_name"], row["content"]), axis=1)
#suffix "none" will be treated as 0
df["suffix_yn"] = df["suffix"].apply(lambda x: 1 if isinstance(x, list) and len(x) > 0 
                                     else (0 if pd.isna(x) or x == "none" else 1))
df["point_process_yn"] = df["point_process"].apply(lambda x: 1 if pd.notna(x) else 0)
df["net_receive_yn"] = df["net_receive"].apply(lambda x: 1 if isinstance(x, list) and len(x) > 0 else (1 if pd.notna(x) else 0))
df["i_nonspecific_yn"] = df["nonspecific_current"].apply(lambda x: 1 if isinstance(x, list) and len(x) > 0 else (1 if pd.notna(x) else 0))
df["not_ion_channel_yn"] = ((df["suffix_yn"] == 0) & 
                          df["read_ion"].isna() & 
                          df["write_ion"].isna() & 
                          df["nonspecific_current"].isna()).astype(int)
df["not_receptor_yn"] = ((df["point_process_yn"] == 0) & 
                          (df["net_receive_yn"] == 0)).astype(int)

df["has_mg_yn"] = df["content"].apply(has_mg)


In [27]:
# Use MultiLabelBinarizer separately for read and write ions
mlb_read = MultiLabelBinarizer()
df_read_exploded = pd.DataFrame(mlb_read.fit_transform(df['read_ion2']), 
                                columns=[f'read_{col}_yn' for col in mlb_read.classes_])

mlb_write = MultiLabelBinarizer()
df_write_exploded = pd.DataFrame(mlb_write.fit_transform(df['write_ion2']), 
                                 columns=[f'write_{col}_yn' for col in mlb_write.classes_])

# Reset index before merging to avoid index mismatches
df = df.reset_index(drop=True)
df_read_exploded = df_read_exploded.reset_index(drop=True)
df_write_exploded = df_write_exploded.reset_index(drop=True)
df = df.drop(columns=['read_ion2', 'write_ion2']).join(df_read_exploded, rsuffix='_read_dup').join(df_write_exploded, rsuffix='_write_dup')

In [28]:
df

,row_id,file_hash,raw_sha,count,url,download_url,content,error_code,exclude_error,exclude_x86,dataset,type,subtype_confidence,notes_free_text,label,new_label,mod_name,suffix,read_ion,write_ion,parameter,state,net_receive,point_process,nonspecific_current,states_count,clamp_yn,suffix_yn,point_process_yn,net_receive_yn,i_nonspecific_yn,not_ion_channel_yn,not_receptor_yn,has_mg_yn,read_ca_i_yn,read_ca_o_yn,read_cl_i_yn,read_cl_o_yn,read_e_ca_yn,read_e_k_yn,read_e_na_yn,read_e_other_yn,read_i_cal_yn,read_i_cl_yn,read_i_k_yn,read_i_na_yn,read_i_other_yn,read_k_i_yn,read_k_o_yn,read_mg_i_yn,read_mg_o_yn,read_na_i_yn,read_na_o_yn,read_other_i_yn,read_other_o_yn,read_unknown_yn,write_ca_i_yn,write_ca_o_yn,write_cl_i_yn,write_cl_o_yn,write_e_ca_yn,write_e_k_yn,write_e_na_yn,write_e_other_yn,write_i_cal_yn,write_i_cl_yn,write_i_k_yn,write_i_mg_yn,write_i_na_yn,write_i_other_yn,write_k_i_yn,write_k_o_yn,write_mg_i_yn,write_na_i_yn,write_na_o_yn,write_other_i_yn,write_other_o_yn,write_unknown_yn
0,1,f8be35d0c20d1b1f3de4c44323e1780ee24f06893b6364...,67b75245b9690ce219f34a5905425672dcb5c0661a885f...,6,https://modeldb.science/183300?tab=2&file=Shor...,https://modeldb.science/getModelFile?model=183...,TITLE Sodium persistent current\n\nCOMMENT\n ...,None,0,0,train,I Na,3 - Highly confident,NaN,i_na_persistent,i_na_persistent,NaP,[INaP],[ena],None,"{'gbar': 0.0, 'ena': 45.0, 'shm': 1.0}",None,None,None,None,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2,e97ca8a7f9734805832e5ae75442d19d3d7796b9a24190...,b45c13164f1eeaf4ba61ee8522304f7382fa0541831b1a...,1,https://modeldb.science/223649?tab=2&file=Altu...,https://modeldb.science/getModelFile?model=223...,TITLE K-A channel from Klee Ficker and Heinema...,None,0,0,train,I K,2 - Mildly confident,STATES n l taul = 0.26*(v+50)/qtl,i_k_a_type_slow,i_k_a,kaprox,[kap],[ek],[ik],"{'sh': 24.0, 'gkabar': 0.008, 'vhalfn': 11.0, ...","[n, l]",None,None,None,2,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
2,3,b428f4a84f4302dd5e9139beb678039621e4ee5e88b780...,b720bf29848dde82785dd101105f185ebcf9affa081dfd...,1,https://modeldb.science/3511?tab=2&file=mcn1/m...,https://modeldb.science/getModelFile?model=351...,INDEPENDENT {t FROM 0 TO 1 WITH 1 (ms)}\n\nNEU...,None,0,0,test,NaN,NaN,NaN,NaN,neither,mcn1_lg,None,None,None,"{'gmax': 0.0, 'e': 0.0}",[synon synoff],None,mcn1_lgsyn,[i],1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,4,606423f8f4a4f406f3387c7ee6f142bd121237272c347d...,81645a02d4dbf324cf8b82e8f31bda2ce46c3753f7a8ac...,1,https://modeldb.science/262115?tab=2&file=demo...,https://modeldb.science/getModelFile?model=262...,TITLE multiple GABAa receptors\n\nCOMMENT\n---...,None,0,0,train,Receptor,3 - Highly confident,NaN,r_gaba_type_a,r_gaba,multiGABAa,None,None,None,"{'Cmax': 0.5, 'Cdur': 0.3, 'Alpha': 20.0, 'Bet...",None,None,multiGABAa,[i],0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,5,99713c0032634e96cc7cde2dce02d8aa2baf75ad4cc835...,5fe32f050754f534d60c7c126ecd805216089cd0594062...,1,https://modeldb.science/150288?tab=2&file=KimE...,https://modeldb.science/getModelFile?model=150...,:Interneuron Cells to Pyramidal Cells GABA wit...,None,0,0,train,Receptor,3 - Highly confident,NaN,r_gaba_general,r_gaba,interV2pyrDDANE_STFD,None,"[eca, ica]",None,"{'srcid': -1.0, 'destid': -1.0, 'Cdur_gaba': 0...",[r_nmda r_gaba capoolcon],[dummy_weight],interV2pyrDDANE_STFD,[igaba],1,0,0,1,1,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,.

# XGB model

In [29]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np


In [30]:
X_train = df[df["dataset"]=="train"].set_index("file_hash")
X_train = X_train.filter(regex=r'(_yn|_count)$')

X_test = df[df["dataset"]=="test"].set_index("file_hash")
X_test = X_test.filter(regex=r'(_yn|_count)$')

y_train = df.loc[df["dataset"] == "train", "new_label"].values

In [31]:

# Encode categorical labels if necessary
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)  # Convert categorical labels to numeric

# Define XGBoost model
xgb_model = xgb.XGBClassifier(
    objective="multi:softmax",  
    num_class=len(label_encoder.classes_),  
    max_depth=6,
    learning_rate=0.1,
    n_estimators=100,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",  # Ensure eval_metric is in the model definition
    random_state=42
)

# Train the model on the full dataset (since no validation is needed)
xgb_model.fit(X_train, y_train_encoded, verbose=True)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_class=19, num_parallel_tree=None, ...)

In [35]:
#Saving
import pickle

# Save XGBoost model
with open("xgb_model.pkl", "wb") as file:
    pickle.dump(xgb_model, file)

In [44]:
X_train.to_csv("X_train.csv")
X_test.to_csv("X_test.csv")

In [49]:
#pd.DataFrame(y_train).to_csv("y_train.csv", index=False)

In [37]:
from sklearn.metrics import accuracy_score, classification_report

# Predict on training data
y_train_pred_enc = xgb_model.predict(X_train)

# Convert predictions back to original labels
y_train_pred = label_encoder.inverse_transform(y_train_pred_enc)

# Accuracy on training data
train_accuracy = accuracy_score(y_train, y_train_pred)
print(f"Training Accuracy: {train_accuracy:.4f}")

# Detailed classification report
print(classification_report(y_train, y_train_pred))


Training Accuracy: 0.6444
                                 precision    recall  f1-score   support

                           i_ca       0.77      0.62      0.69        16
                 i_ca_l_type_ht       0.58      0.75      0.65        20
                 i_ca_t_type_lt       0.75      0.21      0.33        14
i_h_hyperpolarization_activated       0.41      0.79      0.54        14
                            i_k       0.00      0.00      0.00        13
                          i_k_a       0.85      0.48      0.61        23
                         i_k_ca       0.92      1.00      0.96        24
          i_k_delayed_rectifier       0.40      0.84      0.55        25
                     i_k_m_type       0.00      0.00      0.00        11
                         i_leak       0.53      0.75      0.62        12
                           i_na       0.35      0.40      0.38        15
                i_na_persistent       0.75      0.23      0.35        13
                 i_na_tr

/home/imc33/.conda/envs/nn/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/imc33/.conda/envs/nn/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/imc33/.conda/envs/nn/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Compare to GPT

In [60]:
df_gpt = pd.read_csv("../data/mod_files_merged 3.csv")
X_train = X_train.reset_index()

In [65]:
df_gpt_subset = df_gpt[df_gpt["hash"].isin(X_train["file_hash"])].copy()

In [66]:
import pandas as pd
import ast

# List of column names to process
current_cols = ["currents_at_least_1", "currents_at_least_2", "currents_at_least_3"]

# Convert string lists into actual Python lists for all columns
for col in current_cols:
    df_gpt_subset[col] = df_gpt_subset[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Explode all columns, drop NaNs, and extract unique values
unique_currents = pd.concat([df_gpt_subset[col].explode() for col in current_cols]).dropna().unique()

# Convert to a sorted list (optional)
unique_currents_list = sorted(unique_currents)

# Print unique current names
print(unique_currents_list)


['AMPA', 'Ca pump', 'Channelrhodopsin (ChR)', 'D1', 'D2', 'Dopaminergic Receptor', 'Gaba', 'GabaA', 'GabaB', 'I A', 'I A, slow', 'I C', 'I CAN', 'I Ca,p', 'I Calcium', 'I Chloride', 'I Cl, leak', 'I K', 'I K,Ca', 'I K,leak', 'I L high threshold', 'I M', 'I N', 'I Na, leak', 'I Na, slow inactivation', 'I Na,p', 'I Na,t', 'I Potassium', 'I Q', 'I R', 'I Sodium', 'I T low threshold', 'I h', 'I p,q', 'IK Bkca', 'IK Skca', 'I_AHP', 'I_HCO3', 'I_KD', 'I_KHT', 'I_KLT', 'I_Ks', 'I_Na,t', 'KCC2', 'Kainate', 'Kir', 'Kir, inactivating', 'NMDA', 'Na/Ca exchanger', 'Na/K pump', 'P2X3']


In [69]:
import ast
import pandas as pd

def map_current(value):
    """
    Maps individual current names to their corresponding collapsed category.
    
    Args:
        value (str): The name of the current.

    Returns:
        str: The mapped category label.
    """
    if value in ['AMPA']:
        return 'r_glutamate_ampa'
    
    elif value in ['Ca pump']:
        return 'i_ca'
    
    elif value in ['Channelrhodopsin (ChR)']:
        return 'i_other_mixed'
    
    elif value in ['D1', 'D2', 'Dopaminergic Receptor', 'P2X3', 'Kainate']:
        return 'r_other'
    
    elif value in ['Gaba', 'GabaA', 'GabaB']:
        return 'r_gaba'
    
    elif value in ['I A', 'I A, slow']:
        return 'i_k_a'
    
    elif value in ['I C', 'I K', 'I Potassium', 'I KHT', 'I KLT', 'I_Ks']:
        return 'i_k'
    
    elif value in ['I CAN', 'Na/Ca exchanger', 'Na/K pump', 'KCC2']:
        return 'i_other_mixed'
    
    elif value in ['I Ca,p', 'I Calcium', 'I L high threshold', 'I N', 'I Q', 'I R', 'I T low threshold', 'I p,q']:
        return 'i_ca'
    
    elif value in ['I Chloride', 'I Cl, leak']:
        return 'i_cl'
    
    elif value in ['I K,Ca', 'IK Bkca', 'IK Skca', 'I_AHP']:
        return 'i_k_ca'
    
    elif value in ['I K,leak', 'I Na, leak', 'I Cl, leak']:
        return 'i_leak'
    
    elif value in ['I M']:
        return 'i_k_m_type'
    
    elif value in ['I Na, slow inactivation']:
        return 'i_na'
    
    elif value in ['I Na,p']:
        return 'i_na_persistent'
    
    elif value in ['I Na,t', 'I Sodium', 'I_Na,t']:
        return 'i_na_transient'
    
    elif value in ['I h']:
        return 'i_h_hyperpolarization_activated'
    
    elif value in ['NMDA']:
        return 'r_glutamate_nmda'
    
    elif value in ['I_HCO3']:
        return 'i_nonspecific'
    
    elif value in ['I_KD']:
        return 'i_k_delayed_rectifier'
    
    elif value in ['Kir', 'Kir, inactivating']:
        return 'i_k_ir'
    
    return 'neither'  # Default if no match is found


In [72]:
df_gpt_subset

,hash,currents1,notes1,currents2,notes2,currents3,notes3,currents_at_least_1,currents_at_least_2,currents_at_least_3
0,cf53b079807fc6711d9563031aae227dd1a73cf5add973...,[IK Skca],This model uses a Hill function to represent t...,[IK Skca],This model uses a Hill equation to describe th...,[IK Skca],This model uses a Hill function to describe th...,[IK Skca],[IK Skca],[IK Skca]
44,995ba47205bc9c5003009fd6505e94711a57a006823874...,[Ca pump],The model sets up a multi-annulus diffusion sy...,[Ca pump],Includes detailed buffer interaction and radia...,[Ca pump],The model includes diffusion and buffering of ...,[Ca pump],[Ca pump],[Ca pump]
47,58ab31f433aba6ee39f89fd789fba32b740568d56b186e...,[I L high threshold],The model includes a temperature factor for ad...,[I L high threshold],The model uses the Goldman-Hodgkin-Katz (GHK) ...,[I L high threshold],"The model uses a gating variable 'm', calculat...",[I L high threshold],[I L high threshold],[I L high threshold]
86,73efd6a67f1ac3768ba17d96c56e18f2d2e169ec02c8a6...,[I Calcium],The model initializes negative calcium concent...,[I Calcium],Model simulates intracellular calcium dynamics...,[I Calcium],The model reads a calcium current and uses par...,[I Calcium],[I Calcium],[I Calcium]
99,93e2077ddac108534a293d556b63aabc90f913734f1342...,[I T low threshold],The model uses the GHK equation to calculate i...,[I R],The model includes functions for describing th...,[I T low threshold],The model includes a mechanism for modeling ca...,"[I T low threshold, I R]",[I T low threshold],[]
...,...,...,...,...,...,...,...,...,...,...
5057,2506eda78338278ee9944adb440dc621eced205377b831...,[],The model simulates a chirp current for testin...,[],The model primarily deals with generating a ti...,[],This model specifically addresses current inje...,[],[],[]
5066,fdc16c53e18ef0ba78daf28cd1b723a6be60296758de52...,"[I Na,p, I p,q, I Potassium, I Na, leak]",This model implements specific sodium and pota...,"[I Na,p, I Potassium]","The model uses a GHK current equation, which c...","[I Na,p, I Sodium, I Potassium]",The model uses the Goldman-Hodgkin-Katz equati...,"[I Potassium, I Sodium, I Na,p, I Na, leak, I ...","[I Potassium, I Na,p]","[I Potassium, I Na,p]"
5099,30c2b2a64f729f65e4e3f78eb8139101d6b7614a77d254...,[Ca pump],The model incorporates various types of calciu...,[Ca pump],The model includes detailed calcium buffering ...,[Ca pump],The model simulates calcium buffering dynamics...,[Ca pump],[Ca pump],[Ca pump]
5104,b408003421870590bb819f81765f2bd5b9f2f90102b5f3...,[I T low threshold],"The model incorporates two calcium currents, e...",[I T low threshold],Model focuses on Cav3.1 T-type calcium channel...,[I T low threshold],Incorporates temperature dependence of T-type ...,[I T low threshold],[I T low threshold],[I T low threshold]


In [71]:
# Convert string representations of lists into actual Python lists
for col in ["currents1", "currents2", "currents3"]:
    df_gpt_subset[col] = df_gpt_subset[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# Explode lists in all three columns, apply the function, and get unique values
df_gpt_subset["mapped_currents"] = (
    pd.concat([df_gpt_subset[col].explode() for col in ["currents1", "currents2", "currents3"]])
    .dropna()
    .apply(map_current)
    .unique()
)

print(df_gpt_subset["mapped_currents"])


ValueError: Length of values (20) does not match length of index (407)

In [32]:
# Predict on test dataset
y_test_pred_encoded = xgb_model.predict(X_test)

# Convert predictions back to original label names
y_test_pred = label_encoder.inverse_transform(y_test_pred_encoded)

# Convert to a DataFrame (optional)
df_predictions = pd.DataFrame({"Predicted_Label": y_test_pred})
print(df_predictions.head())  # Show first few predictions

    Predicted_Label
0  r_glutamate_ampa
1  r_glutamate_ampa
2           neither
3  r_glutamate_nmda
4           neither


# Flagging Issues

In [ ]:
#row_id: 477 - commented out ranges included (https://modeldb.science/266818?tab=2&file=Ventricular_GUI.CircRes.ModelDB/Kss.mod)
#row_id: 481 - has comments with mod_state variables (https://modeldb.science/267511?tab=2&file=S1_Thal_NetPyNE_Frontiers_2022/sim/mod/ProbAMPANMDA_EMS.mod)
#row_id: 483 - has units in the mod_state ( https://modeldb.science/195666?tab=2&file=DewellGabbiani2018/mod_files/LGMD_KD_ca3.mod)
#row_id: 483 - was only extracting ONE parameter instead of multiple parameters (fixed)
#row_id 31 - has VALENCE in the write_ion (https://modeldb.science/116862?tab=2&file=b09jan13/IL3.mod)
#row_id 99 - need to fix use_ion

# Questions

In [ ]:
#todo: need to get INCLUDE for the notes table
#todo: should we collapse low-frequency labels
#todo: how do we process the SUFFIX, a lot of times the SUFFIX actually is the actual labeL?
#todo: do we include the name
#df["label"].value_counts()


In [ ]:
#REsol;ved Questions
#Is it okay that we make assumptions like start after READ and stop after WRITE, what if there is no WRITE statement?
#How to capture variables that are commented out vs. not? (don't capture stuff commented out)
#what's the best way to capture 


In [ ]:
View(df.loc[481,["url","mod_state"]],50)

In [ ]:
import re





In [36]:
! git add .
! git commit -m "saved xgb"
! git push 

[main e147e62] saved xgb
 2 files changed, 735 insertions(+), 169 deletions(-)
 create mode 100644 code/xgb_model.pkl
Enumerating objects: 8, done.
Counting objects: 100% (8/8), done.
Delta compression using up to 36 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 338.82 KiB | 6.78 MiB/s, done.
Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To github.com:innacohen/mod-extract.git
   6d71a3b..e147e62  main -> main
